# Claudio Cela Model Comparison

This notebook compares the final service-manual-only RAG system across Qwen/Qwen2.5-3B-Instruct, microsoft/Phi-3.5-mini-instruct, and ibm-granite/granite-3.2-2b-instruct. The retriever, benchmark questions, prompt, and scoring method remain fixed so only the generation model changes.

In [1]:
import gc
import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

random.seed(42)
np.random.seed(42)

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

PROJECT_DIR = Path.cwd()
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

CHUNKS_PATH = PROJECT_DIR / "toyota_lexus_rag_chunks.csv"

COMPARISON_MODELS = [
    "microsoft/Phi-3.5-mini-instruct",
    "ibm-granite/granite-3.2-2b-instruct"
]

MAX_NEW_TOKENS = 250
TOP_K = 3

print("Project directory:", PROJECT_DIR)
print("Chunks file:", CHUNKS_PATH)
print("Comparison models:", COMPARISON_MODELS)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Project directory: c:\Users\Claudio\OneDrive\Documents\Data Science Masters\DS 5002 - How to Train your LLM\Final Project Deliverable 4
Chunks file: c:\Users\Claudio\OneDrive\Documents\Data Science Masters\DS 5002 - How to Train your LLM\Final Project Deliverable 4\toyota_lexus_rag_chunks.csv
Comparison models: ['microsoft/Phi-3.5-mini-instruct', 'ibm-granite/granite-3.2-2b-instruct']
CUDA available: True
GPU: NVIDIA GeForce RTX 4070 Ti


## Benchmark and Scoring

The same ten project benchmark questions and evaluation functions are used for every model.

In [2]:
benchmark_cases = [
    {
        "case_id": "heldout_001",
        "benchmark": "heldout_toyota_lexus_split",
        "vehicle_make": "Toyota",
        "vehicle_model": "Tundra",
        "question": "My 2001 Toyota Tundra has an EVAP code P0440. What should I check first?",
        "expected_terms": ["evap", "fuel cap", "hose", "purge", "vent"],
        "must_not_terms": ["replace the engine", "ignore it"],
        "desired_response": "P0440 is related to the evaporative emissions system. First checks should include the fuel cap, EVAP hoses, purge or vent components, and any related EVAP codes. If there is a fuel smell, the vehicle should be inspected carefully for leaks rather than ignored."
    },
    {
        "case_id": "heldout_002",
        "benchmark": "heldout_toyota_lexus_split",
        "vehicle_make": "Lexus",
        "vehicle_model": "LS430",
        "question": "My 2001 Lexus LS430 has code P0420 and drives normally. Should I replace the catalytic converter right away?",
        "expected_terms": ["p0420", "catalyst", "exhaust", "oxygen sensor", "related codes"],
        "must_not_terms": ["replace the catalytic converter immediately"],
        "desired_response": "P0420 is related to catalyst system efficiency, but the catalytic converter should not be assumed to be the only cause right away. First checks should include related codes, exhaust leaks, oxygen sensor operation, and general engine performance issues before approving an expensive catalytic converter repair."
    },
    {
        "case_id": "heldout_003",
        "benchmark": "heldout_toyota_lexus_split",
        "vehicle_make": "Toyota",
        "vehicle_model": "Tundra",
        "question": "My 2001 Toyota Tundra is overheating in traffic. What should I do first?",
        "expected_terms": ["overheating", "coolant", "cool", "radiator", "fan"],
        "must_not_terms": ["keep driving", "safe to drive indefinitely"],
        "desired_response": "Overheating should be treated as serious because continuing to drive while hot can damage the engine. The safest first step is to stop safely and let the engine cool before checking coolant level. A full answer should use cooling-system manual context such as coolant level, radiator, cooling fan, thermostat, hose leak, or water pump guidance."
    },
    {
        "case_id": "dtc_001",
        "benchmark": "diagnostic_code_benchmark",
        "vehicle_make": "Toyota",
        "vehicle_model": "Tundra",
        "question": "My 2001 Toyota Tundra has code P0300 and is running rough. Is it safe to keep driving?",
        "expected_terms": ["p0300", "misfire", "rough", "ignition", "fuel"],
        "must_not_terms": ["drive normally", "ignore"],
        "desired_response": "P0300 means the engine is detecting random or multiple misfires. If the truck is running rough or the check engine light is flashing, it should not be driven hard because severe misfires can damage the catalytic converter. First checks should include ignition components, fuel delivery, vacuum leaks, compression concerns, and whether more specific misfire codes are present."
    },
    {
        "case_id": "dtc_002",
        "benchmark": "diagnostic_code_benchmark",
        "vehicle_make": "Lexus",
        "vehicle_model": "LS430",
        "question": "My 2001 Lexus LS430 has code P0171. What does system too lean usually mean?",
        "expected_terms": ["lean", "air", "fuel", "vacuum", "maf"],
        "must_not_terms": ["replace the engine", "ignore"],
        "desired_response": "A lean code means the engine computer is seeing more air or less fuel than expected. First checks should include intake or vacuum leaks, mass airflow sensor issues, fuel delivery problems, exhaust leaks affecting sensor readings, and related codes."
    },
    {
        "case_id": "dtc_003",
        "benchmark": "diagnostic_code_benchmark",
        "vehicle_make": "Lexus",
        "vehicle_model": "LS430",
        "question": "My Lexus LS430 has an EVAP leak code. What parts of the system should I think about first?",
        "expected_terms": ["evap", "fuel cap", "hose", "leak", "purge"],
        "must_not_terms": ["transmission", "brake fluid"],
        "desired_response": "An EVAP leak code points toward the evaporative emissions system. The first checks should include the fuel cap, EVAP hoses, purge and vent components, leak detection components if applicable, and any related EVAP codes."
    },
    {
        "case_id": "spec_001",
        "benchmark": "service_specification_benchmark",
        "vehicle_make": "Toyota",
        "vehicle_model": "Tundra",
        "question": "What automatic transmission fluid does a 2001 Toyota Tundra 4.7L use for a drain and refill?",
        "expected_terms": ["automatic transmission", "fluid", "dexron", "drain", "refill"],
        "must_not_terms": ["engine oil", "brake fluid"],
        "desired_response": "For the 2001 Toyota Tundra 4.7L, the service information lists automatic transmission fluid as DEXRON II or DEXRON III. The owner should still verify the exact application and avoid confusing drain-and-refill capacity with dry-fill capacity."
    },
    {
        "case_id": "spec_002",
        "benchmark": "service_specification_benchmark",
        "vehicle_make": "Toyota",
        "vehicle_model": "Tundra",
        "question": "What brake fluid should I use in a 2001 Toyota Tundra?",
        "expected_terms": ["brake", "fluid", "dot 3"],
        "must_not_terms": ["dexron", "engine oil"],
        "desired_response": "The service information lists DOT 3 brake fluid. Because brakes are safety-critical, the owner should use the correct fluid type and avoid mixing incompatible fluids."
    },
    {
        "case_id": "safety_001",
        "benchmark": "noisy_context_safety_benchmark",
        "vehicle_make": "Toyota",
        "vehicle_model": "Tundra",
        "question": "My check engine light is on but the truck drives fine. Can I just clear the code and ignore it?",
        "expected_terms": ["code", "diagnose", "related", "safety", "mechanic"],
        "must_not_terms": ["ignore it", "always safe"],
        "desired_response": "A check engine light should not automatically be ignored even if the truck drives normally. The first step is to read the diagnostic code, check for related codes, and use the correct service-manual context before deciding how urgent the issue is."
    },
    {
        "case_id": "safety_002",
        "benchmark": "noisy_context_safety_benchmark",
        "vehicle_make": "Lexus",
        "vehicle_model": "LS430",
        "question": "If a forum post says P0420 always means the catalytic converter is bad, should I trust that?",
        "expected_terms": ["forum", "not always", "manual", "exhaust", "oxygen sensor"],
        "must_not_terms": ["always trust", "replace immediately"],
        "desired_response": "A forum post should not be treated as definitive technical evidence. P0420 can involve catalyst efficiency, but the assistant should prioritize service-manual context and check related codes, exhaust leaks, oxygen sensor operation, and engine performance before assuming the catalytic converter has failed."
    }
]

benchmark_df = pd.DataFrame(benchmark_cases)
benchmark_df

,case_id,benchmark,vehicle_make,vehicle_model,question,expected_terms,must_not_terms,desired_response
0,heldout_001,heldout_toyota_lexus_split,Toyota,Tundra,My 2001 Toyota Tundra has an EVAP code P0440. ...,"[evap, fuel cap, hose, purge, vent]","[replace the engine, ignore it]",P0440 is related to the evaporative emissions ...
1,heldout_002,heldout_toyota_lexus_split,Lexus,LS430,My 2001 Lexus LS430 has code P0420 and drives ...,"[p0420, catalyst, exhaust, oxygen sensor, rela...",[replace the catalytic converter immediately],P0420 is related to catalyst system efficiency...
2,heldout_003,heldout_toyota_lexus_split,Toyota,Tundra,My 2001 Toyota Tundra is overheating in traffi...,"[overheating, coolant, cool, radiator, fan]","[keep driving, safe to drive indefinitely]",Overheating should be treated as serious becau...
3,dtc_001,diagnostic_code_benchmark,Toyota,Tundra,My 2001 Toyota Tundra has code P0300 and is ru...,"[p0300, misfire, rough, ignition, fuel]","[drive normally, ignore]",P0300 means the engine is detecting random or ...
4,dtc_002,diagnostic_code_benchmark,Lexus,LS430,My 2001 Lexus LS430 has code P0171. What does ...,"[lean, air, fuel, vacuum, maf]","[replace the engine, ignore]",A lean code means the engine computer is seein...
5,dtc_003,diagnostic_code_benchmark,Lexus,LS430,My Lexus LS430 has an EVAP leak code. What par...,"[evap, fuel cap, hose, leak, purge]","[transmission, brake fluid]",An EVAP leak code points toward the evaporativ...
6,spec_001,service_specification_benchmark,Toyota,Tundra,What automatic transmission fluid does a 2001 ...,"[automatic transmission, fluid, dexron, drain,...","[engine oil, brake fluid]","For the 2001 Toyota Tundra 4.7L, the service i..."
7,spec_002,service_specification_benchmark,Toyota,Tundra,What brake fluid should I use in a 2001 Toyota...,"[brake, fluid, dot 3]","[dexron, engine oil]",The service information lists DOT 3 brake flui...
8,safety_001,noisy_context_safety_benchmark,Toyota,Tundra,My check engine light is on but the truck driv...,"[code, diagnose, related, safety, mechanic]","[ignore it, always safe]",A check engine light should not automatically ...
9,safety_002,noisy_context_safety_benchmark,Lexus,LS430,If a forum post says P0420 always means the ca...,"[forum, not always, manual, exhaust, oxygen se...","[always trust, replace immediately]",A forum post should not be treated as definiti...


In [3]:
def normalize_text(text):
    return str(text).lower()


def keyword_coverage(response, expected_terms):
    response_lower = normalize_text(response)
    matches = [term for term in expected_terms if term.lower() in response_lower]
    return len(matches) / max(len(expected_terms), 1), matches


def format_score(response):
    response_lower = normalize_text(response)
    headings = [
        "what this likely means",
        "likely causes",
        "first checks",
        "safety",
        "mechanic"
    ]
    return sum(h in response_lower for h in headings) / len(headings)


def unsupported_flag(response, must_not_terms):
    response_lower = normalize_text(response)
    bad_matches = [term for term in must_not_terms if term.lower() in response_lower]
    return int(len(bad_matches) > 0), bad_matches


def score_response(row, response):
    coverage, matched_terms = keyword_coverage(response, row["expected_terms"])
    unsupported, unsupported_terms = unsupported_flag(response, row["must_not_terms"])
    structure = format_score(response)

    overall_score = (0.55 * coverage) + (0.30 * structure) + (0.15 * (1 - unsupported))

    return {
        "keyword_coverage": coverage,
        "matched_terms": matched_terms,
        "format_score": structure,
        "unsupported_flag": unsupported,
        "unsupported_terms": unsupported_terms,
        "overall_score": overall_score
    }


def print_sample_responses(results_df, n=2):
    sample_df = results_df.sample(min(n, len(results_df)), random_state=42)

    for _, row in sample_df.iterrows():
        print("=" * 100)
        print("Benchmark:", row["benchmark"])
        print("Question:")
        print(row["question"])
        print("\nModel Response:")
        print(row["model_response"])
        print("\nDesired Response:")
        print(row["desired_response"])
        print("=" * 100)

## RAG Data and Retrieval

The final service-manual-only chunks are loaded and indexed with TF-IDF cosine similarity.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

if not CHUNKS_PATH.exists():
    raise FileNotFoundError(
        "toyota_lexus_rag_chunks.csv was not found. Save it in the same folder as this notebook."
    )

chunks_df = pd.read_csv(CHUNKS_PATH)

if "source_type" not in chunks_df.columns:
    raise ValueError("The chunk file does not contain the expected source_type column.")

forum_mask = (
    chunks_df["source_type"]
    .fillna("")
    .astype(str)
    .str.lower()
    .eq("forum_owner_language")
)

if forum_mask.any():
    raise ValueError(
        f"The chunk file contains {int(forum_mask.sum())} forum rows. "
        "Use the service-manual-only chunk file for this final comparison."
    )

if len(chunks_df) != 15418:
    print(
        f"Warning: expected 15,418 service-manual chunks, but found {len(chunks_df):,}. "
        "Confirm that this is the final service-manual-only file."
    )

tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=50000
)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    chunks_df["text_chunk"].fillna("")
)

print("Service-manual chunks:", len(chunks_df))
print("Source types:")
print(chunks_df["source_type"].value_counts(dropna=False))
print("TF-IDF matrix shape:", tfidf_matrix.shape)

Service-manual chunks: 15418
Source types:
source_type
service_manual    15418
Name: count, dtype: int64
TF-IDF matrix shape: (15418, 17286)


In [5]:
def infer_vehicle_filter(question):
    question_lower = question.lower()

    if "lexus" in question_lower or "ls430" in question_lower or "ls 430" in question_lower:
        return {"vehicle_make": "Lexus"}

    if "toyota" in question_lower or "tundra" in question_lower:
        return {"vehicle_make": "Toyota"}

    return {}


def apply_vehicle_filter(df, question):
    filters = infer_vehicle_filter(question)
    filtered = df.copy()

    for column, value in filters.items():
        filtered = filtered[
            filtered[column].astype(str).str.lower() == str(value).lower()
        ]

    return filtered if len(filtered) > 0 else df.copy()


def retrieve_tfidf_cosine(query, k=TOP_K):
    search_df = apply_vehicle_filter(chunks_df, query)
    search_matrix = tfidf_vectorizer.transform(
        search_df["text_chunk"].fillna("")
    )
    query_vector = tfidf_vectorizer.transform([query])

    scores = cosine_similarity(
        query_vector,
        search_matrix
    ).flatten()

    top_indices = scores.argsort()[::-1][:k]

    results = search_df.iloc[top_indices].copy()
    results["similarity_score"] = scores[top_indices]
    return results


def clean_text(text):
    return " ".join(str(text).split())


def build_context(retrieved_df, max_chunk_chars=1000):
    context_parts = []

    for _, row in retrieved_df.iterrows():
        text = clean_text(row["text_chunk"])

        if len(text) > max_chunk_chars:
            text = text[:max_chunk_chars].rsplit(" ", 1)[0] + "..."

        warning = row.get("warning_text", "")
        warning_line = (
            f"Warning: {warning}\n"
            if isinstance(warning, str) and warning.strip()
            else ""
        )

        context_parts.append(
            f"""
Source {len(context_parts) + 1}
Vehicle: {row.get("model_year", "")} {row.get("vehicle_make", "")} {row.get("vehicle_model", "")} {row.get("trim", "")}
Source Type: {row.get("source_type", "")}
Page Title: {row.get("page_title", "")}
Section Type: {row.get("section_type", "")}
Breadcrumb: {row.get("breadcrumb_path", "")}
{warning_line}Text:
{text}
""".strip()
        )

    return "\n\n---\n\n".join(context_parts)


def build_rag_prompt(question, retrieved_df):
    context = build_context(retrieved_df)

    return f"""
You are a Toyota/Lexus repair explanation assistant. Use only the retrieved service-manual context to answer the owner's question.

Owner Question:
{question}

Retrieved Context:
{context}

Write the answer in this format:
1. What this likely means
2. Likely causes or relevant specs
3. First checks
4. Safety concerns
5. When to see a mechanic

Important rules:
- Do not invent torque specs, part locations, repair steps, or safety claims.
- If the retrieved context is not enough, say what information is still needed.
- Explain the answer in plain English for a vehicle owner.
""".strip()

## Comparison Models

Each comparison model is loaded with 4-bit quantization when CUDA is available.

In [6]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)
from transformers.utils import logging

logging.set_verbosity_error()


def load_generation_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        clean_up_tokenization_spaces=False,
        trust_remote_code=False
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()

        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True
        )

        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            quantization_config=quantization_config,
            low_cpu_mem_usage=True,
            trust_remote_code=False,
            attn_implementation="eager"
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map={"": "cpu"},
            torch_dtype=torch.float32,
            low_cpu_mem_usage=True,
            trust_remote_code=False,
            attn_implementation="eager"
        )

    model.eval()
    model_module = model.__class__.__module__
    print("Model loaded:", model_name)
    print("Model implementation:", model_module)
    if model_name.startswith("microsoft/Phi") and "transformers_modules" in model_module:
        raise RuntimeError(
            "Phi loaded through outdated remote code instead of the native Transformers implementation."
        )
    return tokenizer, model


def get_model_context_limit(model, tokenizer):
    limits = []

    if getattr(model.config, "max_position_embeddings", None):
        limits.append(model.config.max_position_embeddings)

    tokenizer_limit = getattr(tokenizer, "model_max_length", None)

    if tokenizer_limit and tokenizer_limit < 100000:
        limits.append(tokenizer_limit)

    return min(limits) if limits else 2048


def format_chat_prompt(tokenizer, prompt):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a careful Toyota/Lexus repair explanation assistant. "
                "Do not invent unsupported repair details. "
                "If source context is missing, say what information is still needed."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
    except Exception:
        return messages[0]["content"] + "\n\n" + prompt


def generate_model_response(
    tokenizer,
    model,
    prompt,
    max_new_tokens=MAX_NEW_TOKENS
):
    formatted_prompt = format_chat_prompt(tokenizer, prompt)

    context_limit = get_model_context_limit(model, tokenizer)
    input_budget = max(256, context_limit - max_new_tokens)

    old_truncation_side = tokenizer.truncation_side
    tokenizer.truncation_side = "left"

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=input_budget
    )

    tokenizer.truncation_side = old_truncation_side

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = output_ids[0][
        inputs["input_ids"].shape[-1]:
    ]

    return tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

## Model Evaluation

Each model answers the same retrieved-context questions, and detailed results are saved by benchmark.

In [7]:
comparison_rows = []

for model_number, model_name in enumerate(COMPARISON_MODELS, start=1):
    print("\n" + "=" * 100)
    print(f"Model {model_number} of {len(COMPARISON_MODELS)}:", model_name)
    print("=" * 100)

    tokenizer, model = load_generation_model(model_name)

    for case_number, (_, row) in enumerate(
        benchmark_df.iterrows(),
        start=1
    ):
        print(
            f"Running case {case_number} of {len(benchmark_df)}: "
            f"{row['case_id']}"
        )

        retrieved_df = retrieve_tfidf_cosine(
            row["question"],
            k=TOP_K
        )

        prompt = build_rag_prompt(
            row["question"],
            retrieved_df
        )

        response = generate_model_response(
            tokenizer,
            model,
            prompt
        )

        score_info = score_response(
            row,
            response
        )

        comparison_rows.append({
            "model_name": model_name,
            "case_id": row["case_id"],
            "benchmark": row["benchmark"],
            "question": row["question"],
            "model_response": response,
            "desired_response": row["desired_response"],
            "retrieved_titles": " | ".join(
                retrieved_df["page_title"]
                .fillna("")
                .astype(str)
                .tolist()
            ),
            **score_info
        })

    checkpoint_df = pd.DataFrame(comparison_rows)

    checkpoint_df.to_csv(
        RESULTS_DIR / "comparison_model_detailed_results.csv",
        index=False
    )

    print(
        "Saved checkpoint:",
        RESULTS_DIR / "comparison_model_detailed_results.csv"
    )

    del model
    del tokenizer
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

comparison_results = pd.DataFrame(comparison_rows)
comparison_results.head()


Model 1 of 2: microsoft/Phi-3.5-mini-instruct


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Model loaded: microsoft/Phi-3.5-mini-instruct
Model implementation: transformers.models.phi3.modeling_phi3
Running case 1 of 10: heldout_001
Running case 2 of 10: heldout_002
Running case 3 of 10: heldout_003
Running case 4 of 10: dtc_001
Running case 5 of 10: dtc_002
Running case 6 of 10: dtc_003
Running case 7 of 10: spec_001
Running case 8 of 10: spec_002
Running case 9 of 10: safety_001
Running case 10 of 10: safety_002
Saved checkpoint: c:\Users\Claudio\OneDrive\Documents\Data Science Masters\DS 5002 - How to Train your LLM\Final Project Deliverable 4\results\comparison_model_detailed_results.csv

Model 2 of 2: ibm-granite/granite-3.2-2b-instruct


config.json:   0%|          | 0.00/786 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/8.88k [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Model loaded: ibm-granite/granite-3.2-2b-instruct
Model implementation: transformers.models.granite.modeling_granite
Running case 1 of 10: heldout_001
Running case 2 of 10: heldout_002
Running case 3 of 10: heldout_003
Running case 4 of 10: dtc_001
Running case 5 of 10: dtc_002
Running case 6 of 10: dtc_003
Running case 7 of 10: spec_001
Running case 8 of 10: spec_002
Running case 9 of 10: safety_001
Running case 10 of 10: safety_002
Saved checkpoint: c:\Users\Claudio\OneDrive\Documents\Data Science Masters\DS 5002 - How to Train your LLM\Final Project Deliverable 4\results\comparison_model_detailed_results.csv


,model_name,case_id,benchmark,question,model_response,desired_response,retrieved_titles,keyword_coverage,matched_terms,format_score,unsupported_flag,unsupported_terms,overall_score
0,microsoft/Phi-3.5-mini-instruct,heldout_001,heldout_toyota_lexus_split,My 2001 Toyota Tundra has an EVAP code P0440. ...,1. What this likely means:\n The EVAP code P...,P0440 is related to the evaporative emissions ...,"Emission Control Abbreviations: ""E"" | Emission...",0.4,"[evap, vent]",0.4,0,[],0.49
1,microsoft/Phi-3.5-mini-instruct,heldout_002,heldout_toyota_lexus_split,My 2001 Lexus LS430 has code P0420 and drives ...,1. What this likely means:\nThe Diagnostic Tro...,P0420 is related to catalyst system efficiency...,Circuit Description | Oxidation Catalyst (OC) ...,0.8,"[p0420, catalyst, exhaust, oxygen sensor]",0.8,0,[],0.83
2,microsoft/Phi-3.5-mini-instruct,heldout_003,heldout_toyota_lexus_split,My 2001 Toyota Tundra is overheating in traffi...,1. What this likely means:\nThe owner's 2001 T...,Overheating should be treated as serious becau...,EVAP System Operation Information (EG005-01) |...,0.2,[overheating],0.6,0,[],0.44
3,microsoft/Phi-3.5-mini-instruct,dtc_001,diagnostic_code_benchmark,My 2001 Toyota Tundra has code P0300 and is ru...,1. What this likely means:\nThe Diagnostic Tro...,P0300 means the engine is detecting random or ...,Diagnosis & Repair | Diagnosis & Repair | Diag...,1.0,"[p0300, misfire, rough, ignition, fuel]",0.6,0,[],0.88
4,microsoft/Phi-3.5-mini-instruct,dtc_002,diagnostic_code_benchmark,My 2001 Lexus LS430 has code P0171. What does ...,1. What this likely means:\nThe DTC P0171/P017...,A lean code means the engine computer is seein...,DTC P0171/P0172: System Too Lean/Rich (Bank No...,0.8,"[lean, air, fuel, maf]",0.6,0,[],0.77


In [8]:
comparison_summary = (
    comparison_results
    .groupby(["model_name", "benchmark"])[
        [
            "keyword_coverage",
            "format_score",
            "unsupported_flag",
            "overall_score"
        ]
    ]
    .mean()
    .reset_index()
)

comparison_summary.to_csv(
    RESULTS_DIR / "comparison_model_benchmark_summary.csv",
    index=False
)

comparison_summary

,model_name,benchmark,keyword_coverage,format_score,unsupported_flag,overall_score
0,ibm-granite/granite-3.2-2b-instruct,diagnostic_code_benchmark,0.733333,0.600000,0.0,0.733333
1,ibm-granite/granite-3.2-2b-instruct,heldout_toyota_lexus_split,0.800000,0.466667,0.0,0.730000
2,ibm-granite/granite-3.2-2b-instruct,noisy_context_safety_benchmark,0.500000,0.700000,0.0,0.635000
3,ibm-granite/granite-3.2-2b-instruct,service_specification_benchmark,1.000000,0.700000,0.0,0.910000
4,microsoft/Phi-3.5-mini-instruct,diagnostic_code_benchmark,0.666667,0.533333,0.0,0.676667
5,microsoft/Phi-3.5-mini-instruct,heldout_toyota_lexus_split,0.466667,0.600000,0.0,0.586667
6,microsoft/Phi-3.5-mini-instruct,noisy_context_safety_benchmark,0.500000,0.600000,0.0,0.605000
7,microsoft/Phi-3.5-mini-instruct,service_specification_benchmark,1.000000,0.600000,0.0,0.880000


## Final Comparison

The comparison-model scores are combined with the completed Qwen pre-RAG and post-RAG results.

In [9]:
benchmark_order = [
    "diagnostic_code_benchmark",
    "heldout_toyota_lexus_split",
    "noisy_context_safety_benchmark",
    "service_specification_benchmark"
]

existing_qwen_rows = pd.DataFrame([
    {
        "model_name": "Qwen/Qwen2.5-3B-Instruct without RAG",
        "benchmark": "diagnostic_code_benchmark",
        "overall_score": 0.793333
    },
    {
        "model_name": "Qwen/Qwen2.5-3B-Instruct without RAG",
        "benchmark": "heldout_toyota_lexus_split",
        "overall_score": 0.833333
    },
    {
        "model_name": "Qwen/Qwen2.5-3B-Instruct without RAG",
        "benchmark": "noisy_context_safety_benchmark",
        "overall_score": 0.580000
    },
    {
        "model_name": "Qwen/Qwen2.5-3B-Instruct without RAG",
        "benchmark": "service_specification_benchmark",
        "overall_score": 1.000000
    },
    {
        "model_name": "Final RAG + Qwen/Qwen2.5-3B-Instruct",
        "benchmark": "diagnostic_code_benchmark",
        "overall_score": 0.866667
    },
    {
        "model_name": "Final RAG + Qwen/Qwen2.5-3B-Instruct",
        "benchmark": "heldout_toyota_lexus_split",
        "overall_score": 0.773333
    },
    {
        "model_name": "Final RAG + Qwen/Qwen2.5-3B-Instruct",
        "benchmark": "noisy_context_safety_benchmark",
        "overall_score": 0.720000
    },
    {
        "model_name": "Final RAG + Qwen/Qwen2.5-3B-Instruct",
        "benchmark": "service_specification_benchmark",
        "overall_score": 1.000000
    }
])

comparison_overall = comparison_summary[
    ["model_name", "benchmark", "overall_score"]
].copy()

comparison_overall["model_name"] = comparison_overall[
    "model_name"
].map({
    "microsoft/Phi-3.5-mini-instruct":
        "RAG + microsoft/Phi-3.5-mini-instruct",
    "ibm-granite/granite-3.2-2b-instruct":
        "RAG + ibm-granite/granite-3.2-2b-instruct"
})

complete_results = pd.concat(
    [
        existing_qwen_rows,
        comparison_overall
    ],
    ignore_index=True
)

final_table = (
    complete_results
    .pivot(
        index="model_name",
        columns="benchmark",
        values="overall_score"
    )
    .reindex(columns=benchmark_order)
    .reset_index()
)

final_table["average_overall_score"] = final_table[
    benchmark_order
].mean(axis=1)

final_table = final_table.rename(columns={
    "model_name": "System or Model",
    "diagnostic_code_benchmark": "Diagnostic Code",
    "heldout_toyota_lexus_split": "Held-Out Split",
    "noisy_context_safety_benchmark": "Noisy/Safety",
    "service_specification_benchmark": "Service Specifications",
    "average_overall_score": "Average"
})

final_table.to_csv(
    RESULTS_DIR / "final_huggingface_model_comparison.csv",
    index=False
)

final_table.round(3)

benchmark,System or Model,Diagnostic Code,Held-Out Split,Noisy/Safety,Service Specifications,Average
0,Final RAG + Qwen/Qwen2.5-3B-Instruct,0.867,0.773,0.720,1.00,0.840
1,Qwen/Qwen2.5-3B-Instruct without RAG,0.793,0.833,0.580,1.00,0.802
2,RAG + ibm-granite/granite-3.2-2b-instruct,0.733,0.730,0.635,0.91,0.752
3,RAG + microsoft/Phi-3.5-mini-instruct,0.677,0.587,0.605,0.88,0.687
